# 02 — Baseline: No Spec-Decode vs. vLLM's Built-in EAGLE-3

Runs two conditions against Qwen3-8B (the target model) using `notebooks/common/bench_utils.py`:
1. No speculative decoding (`speculative_config=None`).
2. EAGLE-3 via the published `Tengyunw/qwen3_8b_eagle3` drafter
   (`method="eagle3"`), compatible with vLLM's `qwen3_eagle3.py` model.

Config shape and metrics confirmed in Phase 1 against
`vendor/vllm/examples/features/speculative_decoding/spec_decode_offline.py` — see
`docs/context.md`. `notebooks/03` will add a third condition (our own SGT-QAT drafter)
using the exact same harness, once notebook 01 has produced a real checkpoint.

**Not yet run.** Needs `vendor/vllm` built/installed in this Colab session
(`pip install -e vendor/vllm` or equivalent) plus a GPU large enough for Qwen3-8B.

## Setup

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path('.').resolve()
sys.path.insert(0, str(REPO_DIR / 'notebooks' / 'common'))

import bench_utils
import torch

assert torch.cuda.is_available(), "No GPU detected."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path('.').resolve()
sys.path.insert(0, str(REPO_DIR / 'notebooks' / 'common'))

# TODO: confirm vLLM install approach for this Colab session before running —
# either `pip install -e vendor/vllm` (built from the local clone used for Phase 1
# orientation, pinned to whatever commit that was) or a plain `pip install vllm` if
# reproducing against a specific vendor/vllm commit isn't required for this run.

import bench_utils
import torch

assert torch.cuda.is_available(), "No GPU detected."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
TARGET_MODEL = 'Qwen/Qwen3-8B'
EAGLE3_DRAFTER = 'Tengyunw/qwen3_8b_eagle3'
NUM_SPECULATIVE_TOKENS = 3
MAX_TOKENS = 256
NUM_PROMPTS = 80  # matches spec_decode_offline.py's --test convention; adjust once timing is known

# TODO: swap in a real prompt set (e.g. philschmid/mt-bench, matching
# spec_decode_offline.py's --test convention) before treating results as final —
# placeholder prompts below are only for a first smoke test of the harness.
PROMPTS = [
    "Explain the difference between speculative decoding and beam search.",
    "Write a short function in Python that reverses a linked list.",
    "Summarize the plot of Pride and Prejudice in three sentences.",
] * (NUM_PROMPTS // 3 + 1)
PROMPTS = PROMPTS[:NUM_PROMPTS]

## Run: no speculative decoding (control)

In [ ]:
result_no_spec = bench_utils.run_benchmark(
    run_name='no_spec_decode',
    target_model=TARGET_MODEL,
    prompts=PROMPTS,
    speculative_config=None,
    max_tokens=MAX_TOKENS,
)
bench_utils.save_result(result_no_spec, results_dir=REPO_DIR / 'results')
result_no_spec

## Run: EAGLE-3 baseline

In [ ]:
speculative_config_eagle3 = {
    'method': 'eagle3',
    'model': EAGLE3_DRAFTER,
    'num_speculative_tokens': NUM_SPECULATIVE_TOKENS,
}

result_eagle3 = bench_utils.run_benchmark(
    run_name='baseline_eagle3',
    target_model=TARGET_MODEL,
    prompts=PROMPTS,
    speculative_config=speculative_config_eagle3,
    max_tokens=MAX_TOKENS,
)
bench_utils.save_result(result_eagle3, results_dir=REPO_DIR / 'results')
result_eagle3

## Compare

In [ ]:
bench_utils.summarize([result_no_spec, result_eagle3])

print("\nOnce these numbers look sane, transcribe them into docs/findings.md (methods +")
print("numbers) and note anything surprising in docs/logs.md.")